# Quantum-Guided Cluster Algorithm for Max-Cut

> 🚀 **Skip the local bottleneck.** Qoro is giving away **$100 in free cloud compute credits.**
> Get your API key at **[dash.qoroquantum.net](https://dash.qoroquantum.net)** to run this at scale.

## Why Cloud?

The paper's key result is a **depth sweep** over 28-node, 10-regular graphs at p=1, 2, 3, 5 — each depth is a separate QAOA optimization with its own DE inner loop running hundreds of circuit evaluations. QoroService runs the same Maestro engine you use locally, **GPU-accelerated and dispatched in parallel**, so the full sweep returns in minutes instead of hours.

**Reference:** [arXiv:2508.10656](https://arxiv.org/abs/2508.10656) — Eder et al., Amazon Quantum Solutions Lab

In [ ]:
# Set your API key (get one free at https://dash.qoroquantum.net)
# create a .env file in the repo root with:
#   QORO_API_KEY="your_api_key_here"

import time

from main import run_benchmark

## The Algorithm

Classical methods like simulated annealing make small, local moves that easily get trapped in rugged energy landscapes. The QGCA solves this by using **quantum-derived correlations** to guide cluster formation.

1. **Quantum Phase:** QAOA extracts pairwise correlations ⟨Z_i Z_j⟩ — how spins tend to align in good solutions
2. **Classical Phase:** Cluster Monte Carlo uses these correlations to build meaningful spin clusters, enabling large, coordinated moves through the search space

### Key Divi Features

| Feature | Role |
|---------|------|
| **QAOA** with `.MAXCUT` | Extracts two-point correlations |
| **QWC Observable Grouping** | Up to **60% circuit reduction** |
| **QDrift Trotterization** | Randomized Trotter for shallower circuits at high depth (used in Phase 2) |
| **QoroService** | Cloud backend for 28+ qubit QAOA |

---

## Phase 1 — Local Toy Problem (10 nodes)

Small graph to prove the algorithm works. Runs in seconds on any laptop.

In [ ]:
t0 = time.time()
results_local = run_benchmark(
    n_nodes=10,
    degree=6,
    qaoa_depths=[1, 2],
    n_iterations_factor=200,
    n_repetitions=10,
    lambda_scale=4,
    seed=42,
    use_cloud=False,
    shots=5_000,
    output_dir="plots",
)
phase1_time = time.time() - t0
print(f"\n⏱️ Phase 1 completed in {phase1_time:.1f}s")

---

## Phase 2 — Paper Benchmark with QoroService (28 nodes)

The paper's primary benchmark: **28-node, 10-regular graphs** at depths p=1, 2, 3, 5. Each QAOA run is dispatched to QoroService, and the deeper runs use **QDrift** trotterization — randomized Trotter sampling that produces shallower circuits at high p without sacrificing the exact dynamics in expectation.

**Requirements:**
Create a `.env` file in the repo root:
```
QORO_API_KEY="your_api_key_here"
```
👉 **[Get your free API key →](https://dash.qoroquantum.net)**

In [ ]:
print("☁️  Routing 28-qubit QAOA circuits to Qoro Maestro...")

t0 = time.time()
results_cloud = run_benchmark(
    n_nodes=28,
    degree=10,
    qaoa_depths=[1, 2, 3, 5],
    use_qdrift=True,                  # randomized Trotter — shallower circuits at p=5
    n_iterations_factor=500,
    n_repetitions=30,
    lambda_scale=4,
    seed=42,
    use_cloud=True,
    shots=10_000,
    output_dir="plots",
)
phase2_time = time.time() - t0

print(f"\n⚡ Local  (Phase 1): {phase1_time:.1f}s for 10 nodes")
print(f"⚡ Cloud  (Phase 2): {phase2_time:.1f}s for 28 nodes")

---

## Variation — Swap QAOA for PCE

The cluster algorithm only needs a **correlation matrix**; it doesn't care whether that matrix came from QAOA, [Pauli Correlation Encoding](https://arxiv.org/abs/2401.09421), or anywhere else. Both Divi extractors return the same `CorrelationResult`, so switching is a one-line change.

PCE compresses N variables into far fewer qubits — useful when N grows beyond what direct QAOA can handle:

- **Dense:** ⌈log₂(N+1)⌉ qubits  (16 vars → 5 qubits)
- **Poly:** ⌈√(2N)⌉ qubits        (16 vars → 6 qubits)

`run_benchmark` accepts `pce_encodings` alongside `qaoa_depths`, putting both kinds of source on the same comparison plots.

In [ ]:
results_pce = run_benchmark(
    n_nodes=16,
    degree=10,
    qaoa_depths=[2],                  # one QAOA reference at the transition depth
    pce_encodings=["dense", "poly"],  # plus both PCE encodings on the same plots
    n_iterations_factor=500,
    n_repetitions=20,
    lambda_scale=4,
    seed=42,
    use_cloud=False,
    shots=10_000,
    output_dir="plots_pce",
)

---

## Ready to scale up?

Bigger graphs, deeper circuits, more restarts — all painful sequentially on a laptop, all painless when QoroService dispatches Maestro runs in parallel on GPUs.

👉 **[Get your free API key at dash.qoroquantum.net](https://dash.qoroquantum.net)** — $100 in credits, no credit card required.